# Deploy Sarvam **OCR 3B** Vision Model Package from AWS Marketplace

This sample notebook shows you how to deploy [OCR 3B](https://aws.amazon.com/marketplace/) <!-- TODO: replace with the marketplace listing URL --> — Sarvam AI's document OCR model — using Amazon SageMaker, and how to run all three supported inference modes:

1. **Real-time inference** — POST raw PDF bytes to a SageMaker endpoint and get back a zip archive with the recognized document as Markdown plus per-page layout metadata. Best for small documents that fit the real-time limits (payload ≤ 6 MB, response within 60 seconds).
2. **Asynchronous inference** — upload the PDF to S3 and let the endpoint process it in the background (payloads up to 1 GB, processing up to 1 hour). Best for large or many-page documents.
3. **Batch transform** — OCR a folder of PDFs from S3 in one offline job.

The request/response schema is the same in all three modes: the request body is the raw PDF (`application/pdf`), the response is a zip archive (`application/zip`).

> **Note**: This is a reference notebook and it cannot run unless you make the changes suggested in the notebook (at minimum, set `model_package_arn`).

## Pre-requisites

1. This notebook contains elements which render correctly in the Jupyter interface. Open this notebook from an Amazon SageMaker Notebook Instance or Amazon SageMaker Studio.
1. Ensure that the IAM role used has **AmazonSageMakerFullAccess**.
1. To deploy this ML model successfully, ensure that one of the following applies:
    1. Your IAM role has these three permissions and you have authority to make AWS Marketplace subscriptions in the AWS account used:
        1. **aws-marketplace:ViewSubscriptions**
        1. **aws-marketplace:Unsubscribe**
        1. **aws-marketplace:Subscribe**
    1. or your AWS account has a subscription to the listing. If so, skip step: [Subscribe to the model package](#1.-Subscribe-to-the-model-package).
1. Real-time and asynchronous inference each use one `ml.g6e.xlarge` instance (1× NVIDIA L40S GPU) — make sure your account has quota for `ml.g6e.xlarge for endpoint usage` (the notebook runs one endpoint at a time). Batch transform additionally needs `ml.g6.xlarge for transform job usage` quota (transform quotas default to 0 in new accounts; request an increase via Service Quotas / AWS Support).
1. The notebook works with both major versions of the SageMaker Python SDK — v3 (preinstalled on current SageMaker Studio images) and v2. All SageMaker API calls are made through `boto3`.

## Contents

1. [Subscribe to the model package](#1.-Subscribe-to-the-model-package)
2. [Create an endpoint and perform real-time inference](#2.-Create-an-endpoint-and-perform-real-time-inference)
3. [Asynchronous inference](#3.-Asynchronous-inference)
4. [Perform batch inference](#4.-Perform-batch-inference)
5. [Clean-up](#5.-Clean-up)

## Usage instructions

You can run this notebook one cell at a time (by using Shift+Enter for running a cell).

## 1. Subscribe to the model package

To subscribe to the model package:
1. Open the model package listing page [OCR 3B](https://aws.amazon.com/marketplace/). <!-- TODO: replace with the marketplace listing URL -->
1. On the AWS Marketplace listing, click on the **Continue to subscribe** button.
1. On the **Subscribe to this software** page, review and click on **"Accept Offer"** if you and your organization agree with EULA, pricing, and support terms.
1. Once you click on the **Continue to configuration** button and then choose a **region**, you will see a **Product Arn** displayed. This is the model package ARN that you need to specify while creating a deployable model using boto3. Copy the ARN corresponding to your region and specify it in the following cell.

In [ ]:
model_package_arn = "<Customer to specify Model package ARN corresponding to their AWS region>"

In [ ]:
import io
import json
import os
import time
import uuid
import zipfile

import boto3

try:  # SageMaker Python SDK v3
    from sagemaker.core.helper.session_helper import Session, get_execution_role
except ImportError:  # SageMaker Python SDK v2
    from sagemaker import Session, get_execution_role

role = get_execution_role()
sagemaker_session = Session()
bucket = sagemaker_session.default_bucket()
region = sagemaker_session.boto_region_name
runtime = boto3.client("sagemaker-runtime", region_name=region)
sm_client = boto3.client("sagemaker", region_name=region)
s3_client = boto3.client("s3", region_name=region)
bucket

The next cell defines the names, instance types, and content types used throughout the notebook. The model takes the raw PDF as the request body — there are no other request parameters.

In [ ]:
model_name = "ocr3b-vision"

real_time_inference_instance_type = "ml.g6e.xlarge"  # 1x NVIDIA L40S GPU
batch_transform_inference_instance_type = "ml.g6.xlarge"

content_type = "application/pdf"  # request body: raw PDF bytes
accept = "application/zip"        # response body: zip archive with the OCR results

## 2. Create an endpoint and perform real-time inference

If you want to understand how real-time inference with Amazon SageMaker works, see the [documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/how-it-works-hosting.html).

### A. Create an endpoint

In [ ]:
# Create a deployable model from the model package.
sm_client.create_model(
    ModelName=model_name,
    PrimaryContainer={"ModelPackageName": model_package_arn},
    ExecutionRoleArn=role,
    EnableNetworkIsolation=True,  # marketplace model packages run network-isolated
)

# Deploy the model. Endpoint creation takes roughly 10-15 minutes
# (GPU instance provisioning + container image + model weights download).
sm_client.create_endpoint_config(
    EndpointConfigName=model_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "InstanceType": real_time_inference_instance_type,
            "InitialInstanceCount": 1,
        }
    ],
)
sm_client.create_endpoint(EndpointName=model_name, EndpointConfigName=model_name)
print(f"Creating endpoint {model_name} (takes roughly 10-15 minutes)...")
sm_client.get_waiter("endpoint_in_service").wait(EndpointName=model_name)
print("Endpoint is InService")

Once the endpoint has been created, you can perform real-time inference.

### B. Prepare the input

The request body is simply the **raw bytes of the PDF** — the equivalent of:

```bash
curl -X POST https://<endpoint>/invocations \
     -H "Content-Type: application/pdf" -H "Accept: application/zip" \
     --data-binary @document.pdf -o result.zip
```

The repository ships a two-page sample document (`data/input/sample_document.pdf`) with headings, paragraphs, a list, and a table. Real-time requests must stay within SageMaker's limits: **payload ≤ 6 MB** and **response within 60 seconds** — use [asynchronous inference](#3.-Asynchronous-inference) for anything larger.

In [ ]:
pdf_path = "data/input/sample_document.pdf"
with open(pdf_path, "rb") as f:
    pdf_bytes = f.read()
print(f"{pdf_path}: {len(pdf_bytes)} bytes")

### C. Perform real-time inference

In [ ]:
response = runtime.invoke_endpoint(
    EndpointName=model_name,
    ContentType=content_type,
    Accept=accept,
    Body=pdf_bytes,
)
result_zip = response["Body"].read()
assert result_zip[:2] == b"PK", "expected a zip archive"
print(f"Received {len(result_zip)} bytes ({response['ContentType']})")

### D. Visualize output

The response is a **zip archive** (the expected archive for the sample document ships in `data/output/sample_output.zip`, extracted for browsing under `data/output/extracted/`):

| Archive member | Description |
|---|---|
| `document.md` | The full recognized document as Markdown, pages separated by `---`. Tables are rendered as Markdown tables. |
| `metadata/page_NNN.json` | One file per page: page dimensions and the detected layout blocks. |

Each block in `metadata/page_NNN.json` has:

| Key | Description |
|---|---|
| `block_id` | Unique identifier of the block. |
| `coordinates` | Bounding box `{x1, y1, x2, y2}` in rendered-page pixels (page size in `image_width`/`image_height`). |
| `layout_tag` | Block type, e.g. `headline`, `paragraph`, `ordered-list`, `header`, `footer`. |
| `confidence` | Confidence of the layout detection. |
| `reading_order` | 1-based position of the block in reading order. |
| `text` | Recognized text of the block. |

In [ ]:
archive = zipfile.ZipFile(io.BytesIO(result_zip))
archive.printdir()

document_md = archive.read("document.md").decode("utf-8")
print("\n" + document_md)

In [ ]:
for name in sorted(archive.namelist()):
    if not name.startswith("metadata/"):
        continue
    page = json.loads(archive.read(name))
    print(f"{name}: page {page['page_num']}, "
          f"{page['image_width']}x{page['image_height']} px, {len(page['blocks'])} blocks")
    for block in page["blocks"]:
        preview = block["text"][:60].replace("\n", " ")
        print(f"  #{block['reading_order']:>2} {block['layout_tag']:<12} "
              f"conf={block['confidence']:.2f}  {preview}")

### E. Delete the endpoint

Now that you have successfully performed real-time inference, you do not need the endpoint any more. You can terminate it to avoid being charged.

In [ ]:
sm_client.delete_endpoint(EndpointName=model_name)
sm_client.delete_endpoint_config(EndpointConfigName=model_name)

## 3. Asynchronous inference

[Asynchronous inference](https://docs.aws.amazon.com/sagemaker/latest/dg/async-inference.html) queues requests and processes them in the background: the input PDF is read from S3 and the result zip is written back to S3. Use it for documents that exceed the real-time limits — payloads up to **1 GB** and processing time up to **1 hour** (`InvocationTimeoutSeconds`).

### A. Create an asynchronous endpoint

An endpoint is made asynchronous by its endpoint configuration (`AsyncInferenceConfig`), so we create a second endpoint that reuses the model created in section 2.

In [ ]:
async_endpoint_name = f"{model_name}-async"
async_output_s3 = f"s3://{bucket}/{model_name}/async-output/"

sm_client.create_endpoint_config(
    EndpointConfigName=async_endpoint_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "InstanceType": real_time_inference_instance_type,
            "InitialInstanceCount": 1,
        }
    ],
    AsyncInferenceConfig={
        "OutputConfig": {"S3OutputPath": async_output_s3},
        "ClientConfig": {"MaxConcurrentInvocationsPerInstance": 2},
    },
)
sm_client.create_endpoint(
    EndpointName=async_endpoint_name, EndpointConfigName=async_endpoint_name
)
print(f"Creating endpoint {async_endpoint_name} (takes roughly 10-15 minutes)...")
sm_client.get_waiter("endpoint_in_service").wait(EndpointName=async_endpoint_name)
print("Endpoint is InService")

### B. Invoke the endpoint

Upload the PDF to S3 and pass its location — `invoke_endpoint_async` returns immediately with the S3 locations where the result (`OutputLocation`) or the error message (`FailureLocation`) will appear.

In [ ]:
input_key = f"{model_name}/async-input/sample_document-{uuid.uuid4().hex[:8]}.pdf"
s3_client.put_object(
    Bucket=bucket, Key=input_key, Body=pdf_bytes, ContentType=content_type
)

async_response = runtime.invoke_endpoint_async(
    EndpointName=async_endpoint_name,
    InputLocation=f"s3://{bucket}/{input_key}",
    ContentType=content_type,
    Accept=accept,
    InvocationTimeoutSeconds=3600,
)
print("OutputLocation :", async_response["OutputLocation"])
print("FailureLocation:", async_response.get("FailureLocation"))

### C. Collect the result

Poll S3 until the success object (or the failure object) appears. The success object contains the same zip archive as real-time inference.

In [ ]:
def parse_s3_uri(uri: str):
    bucket_, _, key = uri[len("s3://"):].partition("/")
    return bucket_, key


def wait_for_async_result(response: dict, poll_interval: float = 5.0, timeout: float = 900.0) -> bytes:
    """Poll until the success (or failure) object appears; return the result bytes."""
    locations = [("success", response["OutputLocation"])]
    if response.get("FailureLocation"):
        locations.append(("failure", response["FailureLocation"]))
    deadline = time.time() + timeout

    while time.time() < deadline:
        for label, uri in locations:
            uri_bucket, key = parse_s3_uri(uri)
            try:
                body = s3_client.get_object(Bucket=uri_bucket, Key=key)["Body"].read()
            except s3_client.exceptions.NoSuchKey:
                continue
            if label == "failure":
                raise RuntimeError("Async inference failed: " + body.decode(errors="replace"))
            return body
        time.sleep(poll_interval)

    raise TimeoutError(f"No async result after {timeout:.0f}s")


async_zip = wait_for_async_result(async_response)
print(f"Received {len(async_zip)} bytes\n")
zipfile.ZipFile(io.BytesIO(async_zip)).printdir()

### D. Delete the asynchronous endpoint

In [ ]:
sm_client.delete_endpoint(EndpointName=async_endpoint_name)
sm_client.delete_endpoint_config(EndpointConfigName=async_endpoint_name)

## 4. Perform batch inference

In this section, you will perform batch inference using multiple input payloads together. If you are not familiar with batch transform, and want to learn more, see these links:
1. [How it works](https://docs.aws.amazon.com/sagemaker/latest/dg/ex1-batch-transform.html)
2. [How to run a batch transform job](https://docs.aws.amazon.com/sagemaker/latest/dg/how-it-works-batch.html)

Batch transform sends each S3 object **as-is** as one request body, which matches the model's interface exactly: every input object is one raw PDF. Each input produces one `<object name>.out` file containing the result zip archive.

> **Quota note**: transform-job quotas default to 0 in new accounts. Request `ml.g6.xlarge for transform job usage` ≥ 1 before running this section. Also keep each PDF under the job's `MaxPayloadInMB`.

In [ ]:
# Upload the batch input (every object under the prefix is one raw PDF)
transform_input = sagemaker_session.upload_data("data/input", key_prefix=model_name)
print("Transform input uploaded to " + transform_input)

In [ ]:
# Run the batch-transform job
transform_job_name = f"{model_name}-{uuid.uuid4().hex[:8]}"
transform_output = f"s3://{bucket}/{model_name}/batch-output"

sm_client.create_transform_job(
    TransformJobName=transform_job_name,
    ModelName=model_name,
    MaxPayloadInMB=6,
    TransformInput={
        "DataSource": {
            "S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": transform_input}
        },
        "ContentType": content_type,
    },
    TransformOutput={"S3OutputPath": transform_output, "Accept": accept},
    TransformResources={
        "InstanceType": batch_transform_inference_instance_type,
        "InstanceCount": 1,
    },
)
print(f"Waiting for transform job {transform_job_name}...")
sm_client.get_waiter("transform_job_completed_or_stopped").wait(
    TransformJobName=transform_job_name
)
print("Status:", sm_client.describe_transform_job(
    TransformJobName=transform_job_name)["TransformJobStatus"])

# Output is available on the following path
transform_output

Each input PDF produces one `<object name>.out` file under the output path — the same zip archive as real-time inference.

In [ ]:
out_bucket, _, out_prefix = transform_output[len("s3://") :].partition("/")
obj = s3_client.get_object(Bucket=out_bucket, Key=f"{out_prefix}/sample_document.pdf.out")
batch_zip = obj["Body"].read()
print(f"Received {len(batch_zip)} bytes\n")
zipfile.ZipFile(io.BytesIO(batch_zip)).printdir()

## 5. Clean-up

### A. Delete the model

In [ ]:
sm_client.delete_model(ModelName=model_name)

### B. Unsubscribe to the listing (optional)

If you would like to unsubscribe to the model package, follow these steps. Before you cancel the subscription, ensure that you do not have any [deployable model](https://console.aws.amazon.com/sagemaker/home#/models) created from the model package or using the algorithm. Note — you can find this information by looking at the container name associated with the model.

**Steps to unsubscribe to product from AWS Marketplace**:
1. Navigate to the __Machine Learning__ tab on [__Your Software subscriptions page__](https://aws.amazon.com/marketplace/ai/library?productType=ml).
2. Locate the listing that you want to cancel the subscription for, and then choose __Cancel Subscription__ to cancel the subscription.